# Lab 10 — Auditoría, rendimiento y comparación con pandas

Se ejecuta desde `tu-entrega/` sobre `clinica.duckdb` (ver `sql/carga.sql`). Contiene las tres
partes que pide el enunciado: la auditoría (act. 5), las mediciones de rendimiento (act. 9) y la
comparación con pandas (act. 10).

In [1]:
import re
import statistics
import time

import duckdb
import pandas as pd

import pandas_comparacion as pc

con = duckdb.connect("clinica.duckdb", read_only=True)


def sentencias(ruta):
    """Devuelve las sentencias de un .sql sin comentarios, separadas por ';'."""
    lineas = [l for l in open(ruta, encoding="utf-8") if not l.strip().startswith("--")]
    return [s.strip() for s in "".join(lineas).split(";") if s.strip()]


def mediana_ms(sql, n=31):
    tiempos = []
    for _ in range(n):
        t0 = time.perf_counter()
        con.execute(sql).fetchall()
        tiempos.append((time.perf_counter() - t0) * 1000)
    return statistics.median(tiempos)

## 1. Auditoría (actividad 5)

Cuatro auditorías de `sql/auditoria.sql`. Resultado esperado: 0 filas o 0 %.

In [2]:
nombres = ["5.1 Id duplicados en patients", "5.2 encuentros antes del nacimiento",
           "5.3 observations con valor nulo o vacío", "5.4 encuentros con paciente inexistente"]
for nombre, sql in zip(nombres, sentencias("sql/auditoria.sql")):
    df = con.execute(sql).df()
    print(f"--- {nombre}: {len(df)} fila(s)")
    display(df)

--- 5.1 Id duplicados en patients: 0 fila(s)


,id,veces


--- 5.2 encuentros antes del nacimiento: 0 fila(s)


,encounter_id,patient,encounter_start,patient_birthdate


--- 5.3 observations con valor nulo o vacío: 1 fila(s)


,total_observaciones,nulos_count,nulos_pct,vacios_count,vacios_pct,total_sin_dato_count,total_sin_dato_pct
0,5719327,0,0.0,0,0.0,0,0.0


--- 5.4 encuentros con paciente inexistente: 1 fila(s)


,encuentros_huerfanos
0,0


## 2. Rendimiento (actividad 9)

Consulta 6.15 (urgencias sin diagnóstico). Versión A con `NOT EXISTS`, versión B con
`LEFT JOIN ... IS NULL`. Código y planes completos en `sql/rendimiento.sql`.

In [3]:
A = """WITH urgencias_sin_diagnostico AS (
    SELECT e.Id AS encounter_id, e.patient FROM encounters e
    WHERE e.encounterclass = 'emergency'
      AND NOT EXISTS (SELECT 1 FROM conditions c WHERE c.encounter = e.Id))
SELECT COUNT(*), COUNT(DISTINCT patient) FROM urgencias_sin_diagnostico"""
B = """WITH urgencias_sin_diagnostico AS (
    SELECT e.Id AS encounter_id, e.patient FROM encounters e
    LEFT JOIN conditions c ON c.encounter = e.Id
    WHERE e.encounterclass = 'emergency' AND c.encounter IS NULL)
SELECT COUNT(*), COUNT(DISTINCT patient) FROM urgencias_sin_diagnostico"""
print("A:", con.execute(A).fetchall(), " B:", con.execute(B).fetchall())
print(f"Mediana A (NOT EXISTS):       {mediana_ms(A):.1f} ms")
print(f"Mediana B (LEFT JOIN IS NULL): {mediana_ms(B):.1f} ms")
for nombre, sql in (("A", A), ("B", B)):
    plan = con.execute("EXPLAIN ANALYZE " + sql).fetchall()[0][1]
    operadores = sorted(set(re.findall(r"Join Type: (\w+)", plan)))
    print(f"Plan {nombre}: join {operadores}")

A: [(5272, 1873)]  B: [(5272, 1873)]


Mediana A (NOT EXISTS):       13.1 ms


Mediana B (LEFT JOIN IS NULL): 6.9 ms
Plan A: join ['MARK']
Plan B: join ['RIGHT_ANTI']


Pico de memoria (RSS) medido en procesos nuevos, 1 hilo, 8 corridas idénticas, base de
66 MB: **A = 92 MB (+26), B = 85 MB (+19)**. La hipótesis de que B usaría más memoria resultó falsa.
Explicación completa en `sql/rendimiento.sql`.

## 3. SQL contra pandas (actividad 10)

Tablas cargadas una sola vez a DataFrames; el tiempo de carga se reporta aparte.

In [4]:
_, t, carga = pc.cargar()
p, e, c = t["patients"], t["encounters"], t["conditions"]
print(f"Carga a pandas: {carga * 1000:.0f} ms;  encounters ocupa "
      f"{e.memory_usage(deep=True).sum() / 1e6:.0f} MB con 3 columnas")
display(pc.q62_pandas(p), pc.q66_pandas(p, e), pc.q615_pandas(e, c))

Carga a pandas: 78 ms;  encounters ocupa 98 MB con 3 columnas


,gender,n_pacientes,pct
1,M,2967,50.25
0,F,2937,49.75


,media,mediana,min,max
0,72.77,39.0,1,990


,encuentros_sin_dx,pacientes_afectados
0,5272,1873


In [5]:
sql = {
    "6.2": "SELECT gender, COUNT(*) AS n, ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct "
           "FROM patients GROUP BY gender ORDER BY n DESC",
    "6.6": "WITH pp AS (SELECT p.Id, COUNT(e.Id) n FROM patients p LEFT JOIN encounters e "
           "ON p.Id = e.patient GROUP BY p.Id) SELECT ROUND(AVG(n), 2), MEDIAN(n), MIN(n), MAX(n) FROM pp",
    "6.15": A,
}
py = {"6.2": lambda: pc.q62_pandas(p), "6.6": lambda: pc.q66_pandas(p, e), "6.15": lambda: pc.q615_pandas(e, c)}
lineas = {"6.2": (7, 8), "6.6": (15, 15), "6.15": (16, 11)}  # (SQL, pandas), aproximado
filas = []
for k in sql:
    s, d = mediana_ms(sql[k]), pc.mediana_ms(py[k])
    filas.append({"consulta": k, "DuckDB ms": round(s, 2), "pandas ms": round(d, 2),
                  "pandas/DuckDB": round(d / s, 1), "líneas SQL": lineas[k][0], "líneas pandas": lineas[k][1]})
pd.DataFrame(filas)

,consulta,DuckDB ms,pandas ms,pandas/DuckDB,líneas SQL,líneas pandas
0,6.2,0.80,0.81,1.0,7,8
1,6.6,3.19,24.13,7.6,15,15
2,6.15,13.78,84.22,6.1,16,11


### Lectura

- **Tiempo:** pandas es entre ~6x y ~9x más lento en 6.6 y 6.15; en 6.2 están a la par (<1 ms, ruido).
  6.15 es la más lenta de las tres en pandas en tiempo absoluto (~83 ms). Las razones varían entre
  corridas (en la sesión de medición: 8.2x en 6.6 y 6.0x en 6.15), así que se leen como órdenes de magnitud.
- **Legibilidad:** en 6.15 pandas es más corto, pero el SQL se lee casi como la pregunta clínica.
  En 6.6 el `reindex` de pandas exige pensar en los pacientes sin encuentros, que el `LEFT JOIN` hace explícitos.

### Conclusión (actividad 10)

Para este volumen elijo DuckDB: demostró ser superior o igual en las tres consultas (hasta 8.2x más
rápido en 6.6) y procesa los 429k encuentros en disco sin la sobrecarga de memoria que impone pandas
(98 MB solo en 3 columnas). Cambiaría a pandas únicamente en la última milla si el resultado agregado
(<10k filas) requiere integrarse con scikit-learn o seaborn, o reconsideraría el uso exclusivo de
pandas si el dataset nunca superara los 100k registros y los tiempos se mantuvieran por debajo de 50 ms.